In [18]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from mpl_toolkits.mplot3d import Axes3D
import os

In [21]:
def plot_gmin_noise_surface_smoothed(DATA_PATH, TIME, OUTPUT_DIR='graph/3d'):
    """
    Create a smoothed surface plot showing memory window vs noise and drift scales
    
    Parameters:
    DATA_PATH: str - Path to the data directory
    TIME: str - Time identifier for the CSV file
    OUTPUT_DIR: str - Output directory for saving plots
    """
    
    # Load the data
    csv_path = os.path.join(DATA_PATH, f'inference_results_{TIME}.csv')
    df = pd.read_csv(csv_path)
    
    # Extract the required columns
    drifts = df['drift_scale'].values
    noises = df['noise_scale'].values
    gmins = df['g_min'].values
    norm_accuracies = df['norm_accuracy'].values
    
    # Define grid sizes
    N_noise = 50  # Resolution for noise
    N_drift = 50  # Resolution for drift
    
    # Define grid vectors
    noise_vals = np.linspace(noises.min(), noises.max(), N_noise)
    drift_vals = np.linspace(drifts.min(), drifts.max(), N_drift)
    
    # Initialize surface data arrays
    surface_gmins = np.full((N_drift, N_noise), np.nan)
    
    # Loop over drift values
    for i, fixed_drift in enumerate(drift_vals):
        for j, fixed_noise in enumerate(noise_vals):
            # Define gmin range for this noise and drift
            gmin_range = np.linspace(gmins.min(), gmins.max(), N_noise)
            
            # Create points for interpolation
            points = np.column_stack([noises, drifts, gmins])
            query_points = np.column_stack([
                np.full_like(gmin_range, fixed_noise),
                np.full_like(gmin_range, fixed_drift),
                gmin_range
            ])
            
            # Interpolate norm_accuracy at the fixed noise and drift
            try:
                interpolated_norm_acc = griddata(
                    points, norm_accuracies, query_points, 
                    method='linear', fill_value=np.nan
                )
                
                # Filter valid points (norm_accuracy >= 0.99)
                valid_mask = interpolated_norm_acc >= 0.99
                valid_mask = valid_mask & ~np.isnan(interpolated_norm_acc)
                
                if np.any(valid_mask):
                    # Get the maximum gmin
                    valid_gmins = gmin_range[valid_mask]
                    max_gmin = np.max(valid_gmins)
                    
                    # Store the max gmin in the surface grid
                    surface_gmins[i, j] = max_gmin
                    
            except Exception as e:
                # If interpolation fails, keep NaN
                continue
    
    # Transform the surface data (equivalent to MATLAB's 25 - surface_gmins)
    surface_gmins = 25 - surface_gmins
    
    # Create meshgrid for plotting
    noise_grid, drift_grid = np.meshgrid(noise_vals, drift_vals)
    
    # Create the plot
    fig = plt.figure(figsize=(12, 9))
    ax = fig.add_subplot(111, projection='3d')
    
    # Create surface plot
    surf = ax.plot_surface(
        noise_grid, drift_grid, surface_gmins,
        cmap='viridis',  # Similar to parula colormap
        alpha=0.8,
        edgecolor='gray',
        linewidth=0.1
    )
    
    # Set labels and title
    ax.set_xlabel('Noise Scale', fontsize=12)
    ax.set_ylabel('Drift Scale', fontsize=12)
    ax.set_zlabel('Memory Window (μS)', fontsize=12)
    ax.set_title('One Day', fontsize=14)
    
    # Set limits
    ax.set_ylim([0, 1])
    
    # Invert Z-axis (equivalent to MATLAB's 'ZDir', 'reverse')
    ax.invert_zaxis()
    ax.invert_xaxis()
    
    # Add colorbar
    cbar = plt.colorbar(surf, ax=ax, shrink=0.5, aspect=5)
    cbar.set_label('Memory Window (μS)', rotation=90, fontsize=12)
    
    # Enable grid
    ax.grid(True, alpha=0.5)
    
    # Ensure output directory exists
    #os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # Save the plot
    #output_path = os.path.join(OUTPUT_DIR, f'surface_plot_{TIME}.png')
    #plt.savefig(output_path, dpi=300, bbox_inches='tight')
    
    # Show the plot
    plt.ion()
    plt.show(block=True)
    
    return surface_gmins, noise_vals, drift_vals

def extract_min_max_ranges(drifts, noises, gmins):
    """
    Extract min and max ranges for noise and gmin at each drift level
    
    Parameters:
    drifts, noises, gmins: numpy arrays
    
    Returns:
    drift_levels, noise_min, noise_max, gmin_min, gmin_max: numpy arrays
    """
    # Get unique drift levels
    drift_levels = np.unique(drifts)
    
    # Initialize arrays to store min and max values for each drift level
    noise_min = np.zeros(len(drift_levels))
    noise_max = np.zeros(len(drift_levels))
    gmin_min = np.zeros(len(drift_levels))
    gmin_max = np.zeros(len(drift_levels))
    
    # Loop over each unique drift level
    for i, drift_level in enumerate(drift_levels):
        # Find indices corresponding to the current drift level
        drift_idx = drifts == drift_level
        
        # Extract noise and gmin values for this drift level
        noise_at_drift = noises[drift_idx]
        gmin_at_drift = gmins[drift_idx]
        
        # Compute min and max for noise and gmin at this drift level
        noise_min[i] = np.min(noise_at_drift)
        noise_max[i] = np.max(noise_at_drift)
        gmin_min[i] = np.min(gmin_at_drift)
        gmin_max[i] = np.max(gmin_at_drift)
    
    return drift_levels, noise_min, noise_max, gmin_min, gmin_max


def read_csv_data(csv_path, norm_acc_threshold=0.99,noise_limit=2.0, gmin_limit=15.0):
    """
    Read and filter CSV data
    
    Parameters:
    csv_path: str - Path to CSV file
    norm_acc_threshold: float - Minimum normalized accuracy threshold
    noise_limit: float - Maximum noise scale limit
    gmin_limit: float - Maximum gmin limit
    
    Returns:
    Filtered pandas DataFrame
    """
    df = pd.read_csv(csv_path)
    
    # Apply filters (equivalent to MATLAB's valid_mask)
    valid_mask = (
        (df['norm_accuracy'] >= norm_acc_threshold) &
        (df['noise_scale'] <= noise_limit) &
        (df['g_min'] <= gmin_limit)
    )
    
    return df[valid_mask]


In [22]:
DATA_PATH = '../results'
TIME='day'
TITLE_MAP = {
    'second': '1 Second',
    'hour': '1 Hour',
    'day': '1 Day',
    'week': '1 Week',
    'year': '1 Year',
}
surface_data, noise_vals, drift_vals = plot_gmin_noise_surface_smoothed(
        DATA_PATH, TIME)